In [ ]:
""" Get QoS """
import common_utils
import os
import pandas as pd
import difflib

# Example usage
root_folder = '../../../data_warehouse/minimized_warehouse_6b'
filename = 'worker1.feather'
subfolders = common_utils.find_subfolders_with_file(root_folder, filename)
print(subfolders)
prom_data_paths = {os.path.basename(x): x for x in subfolders}
yolo_data_paths = {key: os.path.join(val, "worker_qos.feather") for key, val in prom_data_paths.items()}


In [ ]:
from matplotlib import pyplot as plt
total_joules_per_model = {}
matches_used = set()
time_lengths = []
for key in prom_data_paths.keys():
    paths = []
    """ Get all workers """
    for work_num in range(1, 6):
        temp_path = os.path.join(prom_data_paths[key], f"worker{work_num}.feather")
        paths.append(temp_path)

    # Plot
    fig = plt.figure(figsize=(10, 6))
    for path in paths:
        df = common_utils.get_cleaned_df(path)
        time_lengths.append(len(df))
        # df = pd.read_feather(path)
        target_word = '"__name__":"node_load5"'
        closest_matches = difflib.get_close_matches(target_word, df.columns, n=2, cutoff=0.05)
        # print(closest_matches)
        # Extract worker number from the path
        worker_num = path.split('worker')[1].split('.')[0]
        # Plot with label
        col = closest_matches[0]
        matches_used.add(col)
        df["time_minutes"] = (df["timestamp"] - df["timestamp"].min()) / 60  # Start time from 0
        df.plot(y=col, x="time_minutes", label=f'Worker {worker_num}', ax=fig.gca())
    # Add legend to show the worker labels
    plt.legend()
    plt.title(key)
    plt.show()
print(f"The following columns were used: {matches_used} (Should be one column only!)")
print(f"The following time lengths were used: {time_lengths}")




In [ ]:
from matplotlib import pyplot as plt
import matplotlib.cm as cm
import numpy as np

total_joules_per_model = {}
matches_used = set()
time_lengths = []

# Define different plot styles for each load metric
plot_styles = {
    "node_load1": {
        "linestyle": "-",
        "marker": "o",
        "color": 'red',
        "linewidth": 2
    },
    "node_load5": {
        "linestyle": "--",
        "marker": "s",
        "color": 'blue',
        "linewidth": 1.5
    }
}

for key in prom_data_paths.keys():
    paths = []
    # Get all workers
    for work_num in range(1, 6):
        temp_path = os.path.join(prom_data_paths[key], f"worker{work_num}.feather")
        paths.append(temp_path)

    # Create a figure with subplots for different load metrics
    fig, axs = plt.subplots(len(plot_styles), 1, figsize=(12, 15), sharex=True)
    fig.suptitle(f'Load Averages - {key}', fontsize=16)

    # Iterate through load metrics
    for idx, target_word in enumerate(plot_styles.keys()):
        ax = axs[idx]

        for path in paths:
            df = common_utils.get_cleaned_df(path)
            time_lengths.append(len(df))

            closest_matches = difflib.get_close_matches(target_word, df.columns, n=2, cutoff=0.05)

            # Extract worker number from the path
            worker_num = path.split('worker')[1].split('.')[0]

            # Get the column and plot style
            col = closest_matches[0]
            matches_used.add(col)

            # Calculate time in minutes
            df["time_minutes"] = (df["timestamp"] - df["timestamp"].min()) / 60

            # Plot with specific style for each load metric
            style = plot_styles[target_word]
            ax.plot(df["time_minutes"], df[col],
                    label=f'Worker {worker_num}',
                    linestyle=style['linestyle'],
                    marker=style['marker'],
                    color=style['color'],
                    linewidth=style['linewidth'])

        ax.set_title(f'{target_word} Load Average')
        ax.set_ylabel('Load')
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.7)

    # Set common x-label
    axs[-1].set_xlabel('Time (minutes)')

    plt.tight_layout()
    plt.show()

print(f"The following columns were used: {matches_used}")
print(f"The following time lengths were used: {time_lengths}")
